In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 22
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.344082295580915

Trial 1 =========================================
18.77235898491488

Trial 2 =========================================
15.926575129018985

Trial 3 =========================================
14.242807573208408

Trial 4 =========================================
14.675898032936882

Trial 5 =========================================
17.582005308980122

Trial 6 =========================================
14.264953175492984

Trial 7 =========================================
15.155465510521541

Trial 8 =========================================
15.227838393283996

Trial 9 =========================================
14.338444853354709

Trial 10 =========================================
14.495723200618427

Trial 11 =========================================
14.373464483980026

Trial 12 =========================================
16.083831584778896

Trial 13 =========================================
15.730331465644712

Trial 14 ========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 33 =========================================
15.942049490761143

Trial 34 =========================================
15.061247530066062

Trial 35 =========================================
13.067934278474382

Trial 36 =========================================
16.0089562884502



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 37 =========================================
15.533942467350524

Trial 38 =========================================
14.06840985218735

Trial 39 =========================================
13.93983943331845

Trial 40 =========================================
14.534812774802692

Trial 41 =========================================
16.92046901284839

Trial 42 =========================================
14.230306817214963

Trial 43 =========================================
15.305009353926584

Trial 44 =========================================
14.01304857802665

Trial 45 =========================================
14.092310640363412

Trial 46 =========================================
13.791165541922028

Trial 47 =========================================
13.457159184434637

Trial 48 =========================================
14.498825192147244

Trial 49 =========================================
14.0433327333213

Trial 50 =========================================
15.558887621875712

Trial 51 ===

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 67 =========================================
15.876076228254572

Trial 68 =========================================
13.609007593904156

Trial 69 =========================================
16.242556047298383

Trial 70 =========================================
13.569133368653901

Trial 71 =========================================
17.987500790984896

Trial 72 =========================================
15.18902079223219

Trial 73 =========================================
13.256390132864954

Trial 74 =========================================
17.564134947370373

Trial 75 =========================================
14.082166002873292

Trial 76 =========================================
13.811849100291516

Trial 77 =========================================
15.367351057169039

Trial 78 =========================================
15.906502124233558

Trial 79 =========================================
14.256829791563057

Trial 80 =========================================
15.95792374454276

Trial 81

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 98 =========================================
14.188022636548785

Trial 99 =========================================
13.751209998138977

[14.3440823  18.77235898 15.92657513 14.24280757 14.67589803 17.58200531
 14.26495318 15.15546551 15.22783839 14.33844485 14.4957232  14.37346448
 16.08383158 15.73033147 13.87925909 14.24582952 15.33135714 15.88948074
 18.40550628 15.93554155 14.96969372 13.99779527 16.26273955 16.04158639
 13.9249829  14.14374318 15.3188434  17.0346838  14.10579664 16.35441977
 14.17218909 13.75777774 14.08501507 15.94204949 15.06124753 13.06793428
 16.00895629 15.53394247 14.06840985 13.93983943 14.53481277 16.92046901
 14.23030682 15.30500935 14.01304858 14.09231064 13.79116554 13.45715918
 14.49882519 14.04333273 15.55888762 13.71143964 16.24645571 13.52745058
 15.05927045 13.85169248 14.73586902 15.82282593 14.97655542 15.29120543
 13.86614142 15.24657784 15.55127887 13.06981286 14.02706308 15.59790302
 13.4249212  15.87607623 13.60900759 16.24255605 13.569

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.77235898491488
Avg = 14.948776550161659
Std = 1.1738440871802074


In [6]:
print(y_max_arr.tolist())

[14.344082295580915, 18.77235898491488, 15.926575129018985, 14.242807573208408, 14.675898032936882, 17.582005308980122, 14.264953175492984, 15.155465510521541, 15.227838393283996, 14.338444853354709, 14.495723200618427, 14.373464483980026, 16.083831584778896, 15.730331465644712, 13.879259087153924, 14.245829523352961, 15.331357142025348, 15.88948074182789, 18.405506275117574, 15.935541546322838, 14.969693722692805, 13.997795266975748, 16.26273955188926, 16.041586394733525, 13.924982895033319, 14.143743183431988, 15.318843395427908, 17.034683802504347, 14.10579664120636, 16.35441976681361, 14.172189094020723, 13.757777738184377, 14.085015068422422, 15.942049490761143, 15.061247530066062, 13.067934278474382, 16.0089562884502, 15.533942467350524, 14.06840985218735, 13.93983943331845, 14.534812774802692, 16.92046901284839, 14.230306817214963, 15.305009353926584, 14.01304857802665, 14.092310640363412, 13.791165541922028, 13.457159184434637, 14.498825192147244, 14.0433327333213, 15.558887621

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-22.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)